<a href="https://colab.research.google.com/github/rebeccaastaix/Dissertation-Supplementary-Materials/blob/main/Qatarv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Qatar_merged_dataset.xlsx to Qatar_merged_dataset.xlsx


In [ ]:
file_name = "Qatar_merged_dataset.xlsx"
sheet_name = "Qatar_Model_Data"

df = pd.read_excel(file_name, sheet_name=sheet_name)

df = df[[
    "Year",
    "Non-Oil Real GDP Growth (%)",
    "Oil Price Change",
    "Qatar Total GDP Growth (annual %)",
    "FDI net inflows (% of GDP)",
    "Non-oil Revenue for General Government (% of Non-oil GDP)",
    "Lag Non-Oil GDP Growth",
    "Lag Total GDP Growth"
]].dropna().copy()

print(df.head())
print(df.columns.tolist())

   Year  Non-Oil Real GDP Growth (%)  Oil Price Change  \
1  2001                     8.036937        -14.323012   
2  2002                     5.126883          1.623926   
3  2003                     5.527286         14.588862   
4  2004                    19.257439         32.851885   
5  2005                     9.487953         43.025704   

   Qatar Total GDP Growth (annual %)  FDI net inflows (% of GDP)  \
1                           3.898187                    1.684982   
2                           7.182152                    3.222105   
3                           3.719959                    2.655416   
4                          19.218915                    3.778180   
5                           7.492758                    5.614130   

   Non-oil Revenue for General Government (% of Non-oil GDP)  \
1                                          15.464114           
2                                          14.631025           
3                                          14.0600

In [ ]:
# total gdp

Y_total = df["Qatar Total GDP Growth (annual %)"]
X_total = df[["Oil Price Change", "Lag Total GDP Growth"]]
X_total = sm.add_constant(X_total)

model_total = sm.OLS(Y_total, X_total).fit()
print(model_total.summary())

                                    OLS Regression Results                                   
Dep. Variable:     Qatar Total GDP Growth (annual %)   R-squared:                       0.411
Model:                                           OLS   Adj. R-squared:                  0.354
Method:                                Least Squares   F-statistic:                     7.315
Date:                               Sat, 18 Apr 2026   Prob (F-statistic):            0.00388
Time:                                       12:33:11   Log-Likelihood:                -76.215
No. Observations:                                 24   AIC:                             158.4
Df Residuals:                                     21   BIC:                             162.0
Df Model:                                          2                                         
Covariance Type:                           nonrobust                                         
                           coef    std err          t      P

In [ ]:
# non oil

Y_nonoil = df["Non-Oil Real GDP Growth (%)"]
X_nonoil = df[["Oil Price Change", "Lag Non-Oil GDP Growth"]]
X_nonoil = sm.add_constant(X_nonoil)

model_nonoil = sm.OLS(Y_nonoil, X_nonoil).fit()
print(model_nonoil.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.244
Model:                                     OLS   Adj. R-squared:                  0.172
Method:                          Least Squares   F-statistic:                     3.381
Date:                         Sat, 18 Apr 2026   Prob (F-statistic):             0.0534
Time:                                 12:33:30   Log-Likelihood:                -82.467
No. Observations:                           24   AIC:                             170.9
Df Residuals:                               21   BIC:                             174.5
Df Model:                                    2                                         
Covariance Type:                     nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------

In [ ]:
# stress tests

oil_shocks = [-10, -20, -30]
results = []

avg_lag_total = df["Lag Total GDP Growth"].mean()
avg_lag_nonoil = df["Lag Non-Oil GDP Growth"].mean()

for shock in oil_shocks:
    pred_total = (
        model_total.params["const"]
        + model_total.params["Oil Price Change"] * shock
        + model_total.params["Lag Total GDP Growth"] * avg_lag_total
    )

    pred_nonoil = (
        model_nonoil.params["const"]
        + model_nonoil.params["Oil Price Change"] * shock
        + model_nonoil.params["Lag Non-Oil GDP Growth"] * avg_lag_nonoil
    )

    results.append([shock, pred_total, pred_nonoil])

stress_df = pd.DataFrame(results, columns=[
    "OilShock_pct",
    "Predicted_Total_GDP_Growth",
    "Predicted_NonOil_GDP_Growth"
])

print(stress_df)

   OilShock_pct  Predicted_Total_GDP_Growth  Predicted_NonOil_GDP_Growth
0           -10                    6.105859                     8.314747
1           -20                    5.376661                     7.957511
2           -30                    4.647464                     7.600276


In [ ]:
# fdi test

Y_ext = df["Non-Oil Real GDP Growth (%)"]
X_ext = df[["Oil Price Change", "FDI net inflows (% of GDP)", "Lag Non-Oil GDP Growth"]]
X_ext = sm.add_constant(X_ext)

model_ext = sm.OLS(Y_ext, X_ext).fit()
print(model_ext.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.524
Model:                                     OLS   Adj. R-squared:                  0.453
Method:                          Least Squares   F-statistic:                     7.337
Date:                         Sat, 18 Apr 2026   Prob (F-statistic):            0.00166
Time:                                 12:34:39   Log-Likelihood:                -76.909
No. Observations:                           24   AIC:                             161.8
Df Residuals:                               20   BIC:                             166.5
Df Model:                                    3                                         
Covariance Type:                     nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------

In [ ]:
# non oil revenue ratio

Y_rev = df["Non-Oil Real GDP Growth (%)"]
X_rev = df[[
    "Oil Price Change",
    "Lag Non-Oil GDP Growth",
    "Non-oil Revenue for General Government (% of Non-oil GDP)"
]]
X_rev = sm.add_constant(X_rev)

model_rev = sm.OLS(Y_rev, X_rev).fit()
print(model_rev.summary())

                                 OLS Regression Results                                
Dep. Variable:     Non-Oil Real GDP Growth (%)   R-squared:                       0.398
Model:                                     OLS   Adj. R-squared:                  0.308
Method:                          Least Squares   F-statistic:                     4.411
Date:                         Sat, 18 Apr 2026   Prob (F-statistic):             0.0155
Time:                                 12:34:56   Log-Likelihood:                -79.722
No. Observations:                           24   AIC:                             167.4
Df Residuals:                               20   BIC:                             172.2
Df Model:                                    3                                         
Covariance Type:                     nonrobust                                         
                                                                coef    std err          t      P>|t|      [0.025      0